# Tutorial 4 — LangGraph: State, Cycles and Agents

**Course:** Text & Language Processing / LLM Practical Track
**Format:** Hands-on notebook
**Suggested duration:** 90–120 minutes
**Prerequisite:** Tutorial 3 (LangChain)

Tutorial 3 ended at a wall. The model asked to call a tool, and nothing ran it — because a chain flows one
way and cannot loop a result back. This notebook removes that limit.

By the end you will be able to:

- express a workflow as a **graph over a shared state**, and say what nodes and edges each do;
- route at runtime with **conditional edges**, including edges that point backwards;
- build a **tool-calling agent** that thinks, acts, observes and thinks again;
- explain why `add_messages` is required, and what breaks without it;
- choose between a chain and a graph for a given problem, with a reason.

## 0. Mental model: a graph is a state machine

LangGraph replaces the pipe with a **graph over a shared state**. Nodes read the state and write to it;
edges decide what runs next, and may be conditional or cyclic:

```text
        ┌─────────────────┐
        ▼                 │
START → agent → (decide) ─┘ → END
          │        │
          └→ tools ┘
```

Three pieces, and everything in this notebook is built from them:

| Piece | What it is |
|---|---|
| **State** | A `TypedDict` holding everything the workflow knows so far — the shared whiteboard. |
| **Node** | A function `state → state`. Does work, returns an updated state. |
| **Edge** | What runs next. Fixed, or chosen at runtime by a function reading the state. |

The practical rule: **a chain if the path is known in advance, a graph if the model decides the path.** An
agent that loops "think → call a tool → look at the result → think again" is a cycle, and a cycle is not
something a pipeline can express.

LangGraph is part of the LangChain ecosystem, not a replacement for it: the models, prompts and chains from
tutorial 3 are exactly what goes *inside* the nodes.

## 1. Setup

Same environment as tutorial 3, plus `langgraph`:

```bash
pip install langchain langchain-core langchain-groq python-dotenv langgraph
```

and the same `.env` beside this notebook:

```text
GROQ_API_KEY=your-key-here
```

The cell below rebuilds the model and the two tools from tutorial 3, section 8, so this notebook stands on
its own.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
assert os.getenv("GROQ_API_KEY"), "GROQ_API_KEY not found — see tutorial 3, section 1."

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool

model = ChatGroq(model="llama-3.1-8b-instant", temperature=0)

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers together."""
    return a * b

@tool
def current_time() -> str:
    """Return the current local time as HH:MM."""
    from datetime import datetime
    return datetime.now().strftime("%H:%M")

tools = [multiply, current_time]
model_with_tools = model.bind_tools(tools)

print("model and", len(tools), "tools ready")

### Code walkthrough — what was carried over

Nothing here is new. `ChatGroq` is the same client, `@tool` the same decorator, and **`.bind_tools(tools)`**
the same call that returns a model advertising them.

Worth re-reading, because the rest of the notebook depends on it: the model **never executes anything**. It
emits a structured request — "call `multiply` with `a=17, b=23`" — on the `tool_calls` attribute of its
reply, and your code decides whether to honour it. That boundary is the security model, and the loop you
build in section 4 is what sits on top of it.

## 2. State, nodes, edges

A graph is built from the three pieces named in section 0. Start with a purely sequential one, so the
machinery is visible before any branching is added.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class GreetState(TypedDict):
    name: str
    age: int
    message: str

def greet(state: GreetState) -> GreetState:
    return {"message": f"Hello, {state['name']}!"}

def add_age(state: GreetState) -> GreetState:
    return {"message": f"{state['message']} You are {state['age']} years old."}

builder = StateGraph(GreetState)
builder.add_node("greet", greet)
builder.add_node("add_age", add_age)

builder.add_edge(START, "greet")
builder.add_edge("greet", "add_age")
builder.add_edge("add_age", END)

graph = builder.compile()

print(graph.invoke({"name": "Maryam", "age": 30})["message"])

### Code walkthrough — state in, partial state out

**`class GreetState(TypedDict)`** declares the shared whiteboard. It is a type annotation, not a container —
LangGraph reads it to know which keys exist.

**A node returns only what it changed.** `greet` returns `{"message": ...}` and says nothing about `name` or
`age`; LangGraph merges that into the running state. Returning the whole state also works, but partial
updates are clearer and are what you will see in real code.

**`START` and `END`** are sentinels marking entry and exit. Every graph needs a path from one to the other.

**`.compile()`** validates the structure — unreachable nodes, missing edges — and returns something with
`.invoke()`. A compiled graph is itself a runnable, so it can sit inside an LCEL chain like any other step.

Run `graph.get_graph().draw_mermaid()` to print a diagram of the structure, which is genuinely useful once
graphs have more than three nodes.

## 3. Conditional edges and loops

The reason to reach for a graph is this section. A **conditional edge** calls a function that inspects the
state and returns the *name* of the next step, which means the path is decided at runtime — and may lead
backwards.

In [ ]:
import random
from typing import List

class GameState(TypedDict):
    guesses: List[int]
    target: int
    attempts: int

def setup(state: GameState) -> GameState:
    return {"guesses": [], "attempts": 0, "target": 7}

def guess(state: GameState) -> GameState:
    return {
        "guesses": state["guesses"] + [random.randint(1, 10)],
        "attempts": state["attempts"] + 1,
    }

def should_continue(state: GameState) -> str:
    if state["guesses"][-1] == state["target"]:
        return "found"
    if state["attempts"] >= 8:
        return "give_up"
    return "retry"

builder = StateGraph(GameState)
builder.add_node("setup", setup)
builder.add_node("guess", guess)

builder.add_edge(START, "setup")
builder.add_edge("setup", "guess")
builder.add_conditional_edges(
    "guess",
    should_continue,
    {"retry": "guess", "found": END, "give_up": END},
)

game = builder.compile()
final = game.invoke({"guesses": [], "target": 0, "attempts": 0})

print("guesses:", final["guesses"])
print("attempts:", final["attempts"])
print("found" if final["guesses"][-1] == final["target"] else "gave up")

### Code walkthrough — the cycle

**`add_conditional_edges(source, router, path_map)`** takes three things: the node the edge leaves, a
function that reads state and returns a **string label**, and a map from labels to destination nodes.

Keeping labels separate from node names is deliberate. The router expresses *a decision* ("retry") while the
map expresses *wiring* ("retry means go back to `guess`"), so you can rewire without touching the logic.

**`"retry": "guess"` points a node back at itself.** That cycle is precisely what a chain cannot express, and
why agents need graphs.

Any loop needs a termination condition that does not depend on the model behaving. Here it is
`attempts >= 8`. In an agent it is a maximum number of tool-calling rounds. Without one, a model that keeps
requesting tools will spin until your budget runs out — the standard failure mode of a first agent.

## 4. A tool-calling agent

Now combine tutorial 3's tools with section 3's cycle. The cycle is: the model thinks; if it asked for tools, run them and return to
the model; otherwise stop. This pattern is called **ReAct** — reason and act.

In [ ]:
from typing import Annotated, Sequence
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

SYSTEM = SystemMessage(
    "You are a helpful assistant. Use the provided tools when they are relevant, "
    "and answer directly when they are not."
)

def call_model(state: AgentState) -> AgentState:
    return {"messages": [model_with_tools.invoke([SYSTEM] + list(state["messages"]))]}

def needs_tools(state: AgentState) -> str:
    return "tools" if state["messages"][-1].tool_calls else "done"

builder = StateGraph(AgentState)
builder.add_node("agent", call_model)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", needs_tools, {"tools": "tools", "done": END})
builder.add_edge("tools", "agent")

agent = builder.compile()

result = agent.invoke({"messages": [HumanMessage("What is 17 times 23, and what time is it?")]})

for m in result["messages"]:
    kind = type(m).__name__
    body = m.content or f"→ calling {[c['name'] for c in m.tool_calls]}"
    print(f"{kind:>12}: {body}")

### Code walkthrough — the agent loop

**`Annotated[Sequence[BaseMessage], add_messages]`** is the one piece of genuinely new syntax, and it is
important. By default a node's return value *replaces* a state key. `add_messages` changes that to
**append**, so each node adds to the conversation rather than overwriting it. Forget this annotation and your
agent loses its history every step — the single most common LangGraph bug.

**`ToolNode(tools)`** is prebuilt: it reads `tool_calls` off the last message, runs the matching functions,
and appends a `ToolMessage` per result. You could write it yourself; there is no reason to.

**`needs_tools`** is the whole control flow. If the last message carries `tool_calls`, go and run them;
otherwise the model produced a final answer, so stop.

**`add_edge("tools", "agent")`** closes the cycle. Results go back to the model, which now sees what the
tools returned and decides again — possibly requesting more tools, possibly answering. The printed transcript
shows the full trace: the request, the results, then the answer.

Compare this to tutorial 3, section 8, where the model asked for a tool and nothing happened. The graph is what turns a
request into an action and back into a conversation.

### Exercise 4.1

Add a `search_wikipedia` tool (`pip install wikipedia`, then call `wikipedia.summary(query, sentences=2)`)
and ask something requiring both search and arithmetic — "How tall is the Eiffel Tower in metres, and what is
that in feet?" Watch the message trace: does the model call the tools in one round or two?

Then add a guard: count rounds in the state and force `END` after three. What does the agent do when a tool
keeps returning something it cannot use?

## 5. Choosing between them

| Situation | Reach for |
|---|---|
| Fixed sequence of steps known in advance | **Chain** |
| Several independent calls on one input | **Chain** with `RunnableParallel` |
| One of N paths, chosen once | **Chain** with `RunnableBranch` |
| Retrieve-then-answer (standard RAG) | **Chain** |
| The model decides what happens next | **Graph** |
| Anything that can repeat a step | **Graph** |
| Multiple cooperating agents, shared state | **Graph** |
| You need to pause, inspect, or resume | **Graph** |

Chains are easier to read, easier to test and easier to debug. **Start with a chain and move to a graph when
you meet a cycle** — not before. A graph built for a pipeline that never branches is just a pipeline with
more ceremony.

The honest summary of the split: LCEL handles *data flow*, LangGraph handles *control flow*. Most real
applications contain both, with chains as the nodes inside a graph.

## Glossary — quick reference

| Term | Meaning |
|---|---|
| **State** | The `TypedDict` a graph passes between nodes — the shared whiteboard. |
| **Node** | A function taking the state and returning the keys it changed. |
| **Edge** | A fixed connection between nodes. |
| **Conditional edge** | An edge whose destination is chosen at runtime by a function reading the state. |
| **Path map** | The dict translating a router's string labels into destination nodes. |
| **`START` / `END`** | Sentinels marking a graph's entry and exit. |
| **`.compile()`** | Validates the graph and returns a runnable, usable inside an LCEL chain. |
| **`add_messages`** | Annotation making a state key append rather than overwrite. Required for conversations. |
| **`ToolNode`** | Prebuilt node that reads `tool_calls`, runs the functions, and appends the results. |
| **ReAct** | The reason-then-act loop: think, call a tool, observe, think again. |
| **Cycle** | An edge pointing back to an earlier node — the thing a chain cannot express. |

---